<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 52%; text-align: right;">
        <a >1</a>
        <a href="locally_deployed_nim.ipynb">2</a>
        <a href="nim_lora_adapter.ipynb">3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <span style="float: left; width: 48%; text-align: right;"><a href="locally_deployed_nim.ipynb">Next Notebook</a></span>
</div>



# Retrieval Augmented Generation (RAG) via NIMs APIs
---

In this notebook, we illustrate how to set up the NVIDIA API Key and demonstrate how to remotely send a simple request to the NVIDIA Inference Microservices(NIM) model. Next, we briefly discussed the Retrieval Augmented Generation(RAG) approach and LangChain framework. Lastly, we demonstrated a simple RAG application using NVIDIA Endpoints. At the end of this notebook:
- you will learn how to set the NVIDIA API key to access NIM models
- understand the concept of RAG 
- learn how to fetch content from a web link, split the content into a text chunk, create embeddings, and save locally with a vector store.
- learn the concept of loading the embeddings from the vector store and build a RAG using NVIDIA Endpoints


## Getting Started

NIMs are quickly accessible via easy-to-use open APIs available at [NVIDIA API Catalog](https://build.nvidia.com/explore/discover), a platform for accessing a wide range of microservices online. To start with NIMs, you need an `NVIDIA API Key` which requires registration. You can register by `clicking on the login button to enter your email address`, as shown in the screenshot below, and follow the rest process or attempt to generate the API Key via the [NVIDIA NGC](https://ngc.nvidia.com/signin) registration (*click on your account name -> setup -> Generate Personal Key*). After completing the process, please save your API Key somewhere you can access for future use. A sample API Key should start with `nvapi-` and 64 other characters, including underscore `_`.

If you already have an account please follow this step to get your NVIDIA API KEY:

- Login to your account from [here](https://build.nvidia.com/explore/discover).
- Click on your model of choice.
- Under Input select the Python tab, and click Get API Key and then click Generate Key.
- Copy and save the generated key as NVIDIA_API_KEY. From there, you should have access to the endpoints.

<div style="text-align: center;">
  <!--<img src="imgs/builder_catalog.jpg" style="width: 800px; height: auto;">-->
  <img src="imgs/nim-catalog.png" style="width: 900px; height: auto;">
</div>


## Setting up NVIDIA API Key

Because we want to access NIM outside the NGC environment, it is important to test our API Key by setting it as an environment variable and using it to send requests to NIM models. Please run the two cells below to test the process.

In [ ]:
import os
import getpass
import warnings
warnings.filterwarnings("ignore")
if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key


In [ ]:
import requests
import json

url = "https://integrate.api.nvidia.com/v1/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer {}".format(os.environ["NVIDIA_API_KEY"])
}
data = {
    "model": "mistralai/mistral-7b-instruct-v0.3",
    "messages": [{"role": "user", "content": "When was Nvidia founded and who is the CEO?"}],
    "temperature": 0.5,
    "top_p": 0.7,
    "max_tokens": 1024,
    "stream": False
}

response = requests.post(url, headers=headers, data=json.dumps(data))

print(response.json()['choices'][0]['message']['content'])

## Retrieval Augmented Generation (RAG)

### What is RAG?

Retrieval-augmented generation (RAG) is a technique for enhancing the accuracy and reliability of generative AI models with facts fetched from external sources.

Large pre-trained language models have been shown to store factual knowledge in their parameters and achieve state-of-the-art results when fine-tuned on downstream NLP tasks. However, their ability to access and precisely manipulate knowledge is still limited, and hence, in knowledge-intensive tasks, their performance lags behind task-specific architectures. Additionally, providing provenance for their decisions and updating their world knowledge remain open research problems. You can learn more [here](https://arxiv.org/pdf/2005.11401) 


#### How does RAG help?
RAG is the architecture that helps users connect the strong semantic capabilities of large language models(LLMs) with their datasets, including metadata, text, images, videos, tables, graphs, etc.  


<div style="text-align: center;">
    <img src="https://www.nvidia.com/en-in/glossary/retrieval-augmented-generation/_jcr_content/root/responsivegrid/nv_container_6840787/nv_image.coreimg.100.630.png/1714589610702/rag-diagram-1920x1080.png" alt="RAG Architecture" style="width: 80%; max-width: 600px;"> <br>
    RAG Architecture
</div>

### RAG Components

A RAG application comprises two principal components: `a retriever` and `a generator`. The retriever (RET) component extracts pertinent information from stored datasets in response to a user query. Subsequently, the generator (GEN) component utilizes the user query with the retrieved information to produce a response.

The architecture of a RAG system primarily consists of the following elements:
- **RET**
    - Embedding Model: A biencoder network that transforms input data into dense vector representations.
    - Vector Store: A database optimized for storing and querying high-dimensional vector embeddings.
    - *Re*-Ranking System (Optional): A cross-encoder that prioritizes retrieved information based on relevance to the query.

- **GEN**
    - Large Language Model (LLM): A neural network trained on vast amounts of text data, capable of generating human-like responses.
    - Guardrails (Optional): Mechanisms implemented to ensure the generated output adheres to predefined constraints and quality standards.

A robust RAG application would need to have  a good:
-  retrieval pipeline
-  generator model and,
-  linking of both

The lab will instantiate all these components one at a time and then bring them together.


## LangChain in Brief

[LangChain](https://python.langchain.com/v0.2/docs/introduction/) is a powerful framework designed to simplify the development of applications using large language models (LLMs). 
For RAG applications, LangChain offers:

- Document loaders for various file types
- Text splitters to chunk documents
- Embeddings to convert text into vector representations
- Vector stores for efficient similarity search
- Retrieval methods to fetch relevant information
- Prompt templates to structure queries and responses

### LLM AI Endpoints

The `langchain-nvidia-ai-endpoints` package contains LangChain integrations building applications with models on NVIDIA NIM inference microservice. The [ChatNVIDIA](https://python.langchain.com/v0.2/docs/integrations/chat/nvidia_ai_endpoints/) class is part of `langchain_nvidia_ai_endpoints,` allowing access to NVIDIA NIM for chat applications and connecting to hosted or locally deployed microservices.
Below, we demonstrate an example using `meta/llama3-8b-instruct`. Each model is uniquely identified via a model name key that can be found in the catalog (e.g.,  meta/llama3-8b-instruct, meta/llama-3.1-70b-instruct)


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

model = "meta/llama3-8b-instruct"
llm = ChatNVIDIA(model=model, max_tokens=1024)

In [ ]:
result = llm.invoke("What is a RAG?")
result

**Likely output**:
```
AIMessage(content='A RAG is an acronym that stands for " Registered Anhydrous Glycerol". Anhydrous means "without water". Glyserol is a type of solvent that is commonly used in various industries such as pharmaceuticals, cosmetics, and food processing.\n\nRegistered Anhydrous Glycerol, or RAG, is a specific type of glycerol that has been purified to remove impurities and water content, making it suitable for use in high-purity applications such as pharmaceutical manufacturing, cosmetics, and food processing.\n\nIn the UK, RAG is a specific grade of glycerol that is certified by the British Pharmacopoeia (BP) and meets certain standards for purity and quality.', response_metadata={'role': 'assistant', 'content': 'A RAG is an acronym that stands for " Registered Anhydrous Glycerol". Anhydrous means "without water". Glyserol is a type of solvent that is commonly used in various industries such as pharmaceuticals, cosmetics, and food processing.\n\nRegistered Anhydrous Glycerol, or RAG, is a specific type of glycerol that has been purified to remove impurities and water content, making it suitable for use in high-purity applications such as pharmaceutical manufacturing, cosmetics, and food processing.\n\nIn the UK, RAG is a specific grade of glycerol that is certified by the British Pharmacopoeia (BP) and meets certain standards for purity and quality.', 'token_usage': {'prompt_tokens': 16, 'total_tokens': 161, 'completion_tokens': 145}, 'finish_reason': 'stop', 'model_name': 'meta/llama3-8b-instruct'}, id='run-b67ddf47-3977-4bbc-b634-8add975879e1-0', role='assistant')
```


You can run the cell below to access list of the available models 

In [ ]:
ChatNVIDIA.get_available_models()

#### Analyzing Response Content 

Let's try to decipher the output components from the previous cell.

In [ ]:
import pprint
from IPython.display import display, Markdown, Latex

Display the main response content

In [ ]:
display(Markdown(result.content))

`response_metadata` gives other key details

In [ ]:
result.response_metadata

#### LLM Response Dictionary Keys

- `role`: Identifies the responder (e.g., 'assistant')
- `content`: The actual text response from the AI
- `token_usage`:
  - `prompt_tokens`: Number of tokens in the input
  - `total_tokens`: Total tokens used in the interaction
  - `completion_tokens`: Number of tokens in the AI's response
- `finish_reason`: Why the AI stopped generating text
- `model_name`: Specifies the AI model used

In [ ]:
# Let's try asking a followup question
result = llm.invoke("How is it helpful?")
display(Markdown(result.content))

**NOTE:** Observe the stateless behvaiour of LLMs, they have no context of our previous query. Thus, it is important we pass all our relevant context, background and queries in a single prompt.

## RAG Application

In this section, we will create a simple RAG application using NIM. The application will take data sources from a web link, read the content as documents, split the documents, instantiate embedding, and store them in a vector database. Furthermore, we will create a [conversational retrieval chain](https://python.langchain.com/v0.1/docs/modules/chains/#conversationalretrievalchain-with-streaming-to-stdout) using two LLMs: one for summarization and another for chat. The importance of the combined LLM is to assist in improving the overall result in complex scenarios. Finally, we will add a question generator to generate relevant query prompts. The procedural steps to follow are itemized below:

- Import all the needed libraries
- Create web link data source 
- Create a function that loads the HTML files from the web link
- Create embeddings and document text splitter 
- Generate embeddings using NVIDIA AI Endpoints from LangChain and save to the offline vector store
- Load the embeddings from the vector store and build a RAG using NVIDIA Endpoints
- Create a conversational retrieval chain using two LLMs
- Add a question generator to generate a relevant query prompt


#### Import libraries

In [ ]:
from langchain.chains import ConversationalRetrievalChain, LLMChain
from langchain.chains.conversational_retrieval.prompts import CONDENSE_QUESTION_PROMPT, QA_PROMPT
from langchain.chains.question_answering import load_qa_chain
from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

#### Create Web Link Data Source

You can replace and add more web links of your choice. 

In [ ]:
urls = ["https://www.nvidia.com/en-in/glossary/retrieval-augmented-generation/",
       "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
        "https://docs.nvidia.com/cuda/",
        "https://github.com/NVIDIA/cuda-samples"
       ]

#### Create A Function To Load HTML Files

Below is a helper function for loading html files, which we’ll use to generate the embeddings. The function will load the relevant HTML documents from our URL data source by creating a [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) object to parse HTML. The Beautiful Soup is a Python library for pulling data from HTML and XML files. We can remove HTML script and style tags through the object and get plain text from the HTML document.


In [ ]:
import re
import requests
from bs4 import BeautifulSoup
from typing import List, Union

def html_document_loader(url: Union[str, bytes]) -> str:
    """
    Loads the HTML content of a document from a given URL and return it's content.

    Args:
        url: The URL of the document.

    Returns:
        The content of the document.

    Raises:
        Exception: If there is an error while making the HTTP request.

    """
    try:
        response = requests.get(url)
        html_content = response.text
    except Exception as e:
        print(f"Failed to load {url} due to exception {e}")
        return ""

    try:
        # Create a Beautiful Soup object to parse html
        soup = BeautifulSoup(html_content, "html.parser")

        # Remove script and style tags
        for script in soup(["script", "style"]):
            script.extract()

        # Get the plain text from the HTML document
        text = soup.get_text()

        # Remove excess whitespace and newlines
        text = re.sub("\s+", " ", text).strip()

        return text
    except Exception as e:
        print(f"Exception {e} while loading document")
        return ""

#### Create Embeddings and Document Text Splitter

Let's create a function that initializes the path to store our embeddings, execute the `html_document_loader` function, and split the document into chunks of text. However, there are things to consider within the function.

- **Key Considerations** 
    - Make sure to pay attention to the chunk_size parameter in [TextSplitter](https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/recursive_text_splitter/).  
    - Setting the right chunk size is critical for RAG performance, as much of a RAG's success is based on the retrieval step finding the right context for generation. 
    - The entire prompt (retrieved chunks + user query) must fit within the [LLM's context window](https://www.appen.com/blog/understanding-large-language-models-context-windows). Therefore, you should not specify chunk sizes that are too big and balance them out with the estimated query size.   
    - For example, while OpenAI LLMs have a context window of 8k-32k tokens, Llama3 is limited to 8k tokens.   
    - Experiment with different chunk sizes, but typical values should be 100-600, depending on the LLM.

   

In [ ]:
def create_embeddings(embeddings_model,embedding_path: str = "./embed"):

    embedding_path = "./embed"
    print(f"Storing embeddings to {embedding_path}")

    documents = []
    for url in urls:
        document = html_document_loader(url)
        documents.append(document)


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=0,
        length_function=len,
    )
    print("Total documents:",len(documents))
    texts = text_splitter.create_documents(documents)
    print("Total texts:",len(texts))
    index_docs(embeddings_model,url, text_splitter, texts, embedding_path,)
    print("Generated embedding successfully")

#### Generate Embeddings Using NVIDIA AI Endpoints From LangChain

In this section we demostrate how to generate embeddings using NVIDIA AI Endpoints for LangChain and save embeddings to offline vector store in the `/embed` directory for future re-use. Create the embeddings model using NVIDIA Retrieval QA Embedding endpoint. This model represents words, phrases, or other entities as vectors of numbers and understands the relation between words and phrases. The following are the key embedding models on the Nvidia Catalog, with more to come in upcoming releases:

* `NV-EmbedQA-E5-v5`: an embedding model optimized for text question-answering retrieval.
* `NV-EmbedQA-Mistral7B-v2`: a multilingual model fine-tuned for text embedding and accurate question answering. 
* `Snowflake-Arctic-Embed-L`: an optimized model for text embedding

<div style="text-align: center;">
    <img src="imgs/architecture-NeMo-Retriever.png" alt="RAG Architecture" style="width: 80%; max-width: 600px;"> <br>
    Embedding's architecture
</div>


In [ ]:
embeddings_model = NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5") # or use NV-Embed-QA

The below flowchart explains the next steps after the embedding model creation:
<div style="text-align: center;">
    <img src="imgs/embeddings_flow.svg" alt="RAG Architecture" style="width: 80%; max-width: 600px;"> <br>
    Embedding's architecture
</div>
<br/>

Below, we create an index_docs function that loops through the document page content to extend text and metadata and applies [FAISS](https://faiss.ai/index.html). The embeddings are stored locally.

In [ ]:
from typing import List, Union
import os
from langchain.vectorstores import FAISS

def index_docs(embeddings_model, url: Union[str, bytes], splitter, documents: List[str], dest_embed_dir: str) -> None:
    """
    Split the documents into chunks and create embeddings for them.
    
    Args:
        embeddings_model: Model used for creating embeddings.
        url: Source url for the documents.
        splitter: Splitter used to split the documents.
        documents: List of documents whose embeddings need to be created.
        dest_embed_dir: Destination directory for embeddings.
    """
    texts = []
    metadatas = []

    for document in documents:
        chunk_texts = splitter.split_text(document.page_content)
        texts.extend(chunk_texts)
        metadatas.extend([document.metadata] * len(chunk_texts))

    if os.path.exists(dest_embed_dir):
        docsearch = FAISS.load_local(
            folder_path=dest_embed_dir, 
            embeddings=embeddings_model, 
            allow_dangerous_deserialization=True
        )
        docsearch.add_texts(texts, metadatas=metadatas)
    else:
        docsearch = FAISS.from_texts(texts, embedding=embeddings_model, metadatas=metadatas)

    docsearch.save_local(folder_path=dest_embed_dir)

#### Load Embeddings from the Vector Store and Build a RAG using NVIDIA Endpoints

Next, we call the function `create_embeddings` and load documents from [vector store](https://developer.nvidia.com/blog/accelerating-vector-search-fine-tuning-gpu-index-algorithms/) using FAISS. The Vector store stores relevant information in a high dimensional space called embeddings. They also act as the junctions for retrieving the most relevant documents to a query during inference time. There are multiple algorithmic approaches to information retrieval that can be found [here](https://developer.nvidia.com/blog/accelerating-vector-search-fine-tuning-gpu-index-algorithms/).

Please run the two cells below. 

In [ ]:
%%time
create_embeddings(embeddings_model=embeddings_model)

In [ ]:
# load Embed documents
embedding_path = "./embed/"
docsearch = FAISS.load_local(folder_path=embedding_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)

#### Create A Conversational Retrieval Chain Using Two LLMs

Since NIMs are deployed as standalone services, multiple NIMs can be utilized for different components. We would showcase a chain would use two different LLMs:  
- `Llama3.1-70B-Instruct` for summarization of the conversation before the current message
- `Llama3-8B-Instruct` For responding to chat message

Since each model has its strengths, individual NIMs can be allocated to those areas. For example, a model with a large context window can be used for conversation history and summarisation, while another model with function-calling capabilities can be used for task executions.

Please run the cell below to execute the chain.


In [ ]:
# load our models
summary_llm = ChatNVIDIA(model="meta/llama-3.1-70b-instruct")
chat_llm = ChatNVIDIA(model="meta/llama3-8b-instruct",
                      temperature=0.1,
                      max_tokens=1000,
                      top_p=1.0)

In [ ]:
from langchain.chains import (
create_history_aware_retriever,
create_retrieval_chain)

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory


For a conversational chain, we need to keep response grounded and related to the past queries of the session. Hence, we define prompt templates that make the LLM aware of the added context and response limitations

In [ ]:
#Setup Message History format and retriever
contextualize_q_system_prompt = """Given a chat history and the latest user question which might reference context in the chat history, formulate a standalone question which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

#Setup Agent Behaviour
qa_system_prompt = """You are an assistant for question-answering tasks. \
Try to use the following pieces of retrieved context to answer the question. \
Use three sentences maximum and keep the answer concise.\

{context}"""
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

Create retriever and establish the chain


In [ ]:
history_aware_retriever = create_history_aware_retriever(
    summary_llm, 
    docsearch.as_retriever(), # the vectorstore serves as the junction to retrieve documents with highest similarity to the query.
    contextualize_q_prompt
)
question_answer_chain = create_stuff_documents_chain(chat_llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [ ]:
# sample session creation
store = {}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

#### Test With Query

In [ ]:
conversational_rag_chain.invoke(
    {"input": "What is RAG?"},
     config={
        "configurable": {"session_id": "abc123"}
    }
)["answer"]

In [ ]:
out = conversational_rag_chain.invoke(
    {"input": "How is it helpful?"},
     config={
        "configurable": {"session_id": "abc123"}
    }
)["answer"]
print(out)

In [ ]:
out = conversational_rag_chain.invoke(
    {"input": "What is the meaning of retrieval?"},
     config={
        "configurable": {"session_id": "abc123"}
    }
)["answer"]
print(out)

In [ ]:
out = conversational_rag_chain.invoke(
    {"input": "What is CUDA?"},
     config={
        "configurable": {"session_id": "xyz456"}
    }
)["answer"]

print(out)

In [ ]:
out = conversational_rag_chain.invoke(
    {"input": "Can you write a kernel to add the elements of two arrays and store the output in a third array?"},
     config={
        "configurable": {"session_id": "xyz456"}
    }
)["answer"]

print(out)

Let’s proceed to the next notebook to create a similar application with locally deployed NIM.

---

## References

- https://developer.nvidia.com/blog/tips-for-building-a-rag-pipeline-with-nvidia-ai-langchain-ai-endpoints/
- https://nvidia.github.io/GenerativeAIExamples/latest/notebooks/05_RAG_for_HTML_docs_with_Langchain_NVIDIA_AI_Endpoints.html

## Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.


<br>
<div>
    <span style="float: left; width: 52%; text-align: right;">
       <a>1</a>
        <a href="locally_deployed_nim.ipynb">2</a>
        <a href="nim_lora_adapter.ipynb">3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <span style="float: left; width: 48%; text-align: right;"><a href="locally_deployed_nim.ipynb">Next Notebook</a></span>
</div>

<br>
<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>
